In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras.models import Model, load_model
from keras.layers import Input,Embedding,GRU,Dense,Bidirectional,Dropout, Concatenate,Attention
from keras.optimizers import Adam
from keras.callbacks import LearningRateScheduler
import tensorflow as tf
import pickle

In [2]:
def load_data(path):
    df = pd.read_parquet(path)
    translations = df['translation'].apply(lambda x: x if isinstance(x,dict) else eval(x))
    eng_sentences = [x['en'] for x in translations]
    hi_sentences = [x['hi'] for x in translations]
    return eng_sentences,hi_sentences

In [3]:
def tokenize_and_pad(sentences, num_words=None,max_len=None):
    tokenizer = Tokenizer(num_words=num_words,filters='')
    tokenizer.fit_on_texts(sentences)
    tensor = tokenizer.texts_to_sequences(sentences)
    tensor = pad_sequences(tensor,padding='post',maxlen=max_len)
    return tensor, tokenizer

In [4]:
def prepare_decoder_data(target_tensor):
    decoder_input = target_tensor[:,:-1]
    decoder_output = target_tensor[:,1:]
    return decoder_input,decoder_output

In [5]:
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        decoder_output, encoder_output = inputs

        score = tf.matmul(decoder_output, encoder_output, transpose_b=True)
        attention_weights = tf.nn.softmax(score, axis=-1)
        context_vector = tf.matmul(attention_weights, encoder_output)
        return context_vector

    def get_config(self):
        return super().get_config()

    

In [6]:
def build_model(input_vocab_size, target_vocab_size, embedding_dim, units, max_encoder_len, max_decoder_len):
    encoder_inputs = Input(shape=(max_encoder_len,))
    enc_emb = Embedding(input_vocab_size, embedding_dim)(encoder_inputs)
    encoder_output, forward_h, backward_h = Bidirectional(GRU(units, return_sequences=True, return_state=True))(enc_emb)
    encoder_state = Concatenate()([forward_h, backward_h])

    decoder_inputs = Input(shape=(max_decoder_len,))
    dec_emb = Embedding(target_vocab_size, embedding_dim)(decoder_inputs)
    decoder_gru = GRU(units * 2, return_sequences=True)
    decoder_output = decoder_gru(dec_emb, initial_state=encoder_state)

    attention = AttentionLayer()
    context_vector = attention([decoder_output, encoder_output])
    combined = Concatenate()([decoder_output, context_vector])

    dropout = Dropout(0.3)(combined)
    dense_output = Dense(target_vocab_size, activation='softmax')(dropout)

    model = Model([encoder_inputs, decoder_inputs], dense_output)
    return model

In [7]:
def scheduler(epoch, lr):
    if epoch < 5:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

In [8]:
eng_sentences, hi_sentences = load_data('data\eng_hi_data.parquet')
hi_sentences = ['<sos> ' + sent + ' <eos>' for sent in hi_sentences]

In [9]:
max_vocab_size = 30000
# Calculate dynamic max_len
eng_max_len = max(len(sentence.split()) for sentence in eng_sentences)
hi_max_len = max(len(sentence.split()) for sentence in hi_sentences)

# Add space for <sos> and <eos> tokens in Hindi
hi_max_len += 2

# Optional: cap at a reasonable max to avoid memory overload
eng_max_len = min(eng_max_len, 50)
hi_max_len = min(hi_max_len, 50)


In [10]:
input_tensor,input_tokenizer = tokenize_and_pad(eng_sentences,num_words=max_vocab_size, max_len=eng_max_len)
target_tensor,target_tokenizer = tokenize_and_pad(hi_sentences,num_words=max_vocab_size,max_len=hi_max_len)

In [11]:
input_vocab_size = min(max_vocab_size,len(input_tokenizer.word_index)+1)
target_vocab_size = min(max_vocab_size, len(target_tokenizer.word_index)+1)

In [12]:
input_train,input_val,target_train,target_val = train_test_split(input_tensor,target_tensor,test_size=0.2)

In [13]:
decoder_input_train,decoder_output_train = prepare_decoder_data(target_train)
decoder_input_val, decoder_output_val = prepare_decoder_data(target_val)

In [14]:
model = build_model(
    input_vocab_size=input_vocab_size,
    target_vocab_size=target_vocab_size,
    embedding_dim=128,
    units=128,
    max_encoder_len=eng_max_len,
    max_decoder_len=hi_max_len-1
)

In [15]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 50)]         0           []                               
                                                                                                  
 embedding (Embedding)          (None, 50, 128)      3840000     ['input_1[0][0]']                
                                                                                                  
 input_2 (InputLayer)           [(None, 49)]         0           []                               
                                                                                                  
 bidirectional (Bidirectional)  [(None, 50, 256),    198144      ['embedding[0][0]']              
                                 (None, 128),                                                 

In [16]:
model.fit(
    [input_train,decoder_input_train],
    decoder_output_train,
    validation_data=([input_val,decoder_input_val],decoder_output_val),
    epochs=2,
    batch_size=32,
    callbacks=[LearningRateScheduler(scheduler)]
)

Epoch 1/2
41478/41478 [==============================] - 4653s 112ms/step - loss: 1.1173 - accuracy: 0.8200 - val_loss: 0.8517 - val_accuracy: 0.8464 - lr: 0.0010
Epoch 2/2
41478/41478 [==============================] - 4605s 111ms/step - loss: 0.8416 - accuracy: 0.8476 - val_loss: 0.7768 - val_accuracy: 0.8562 - lr: 0.0010


In [17]:
model.save("eng_hi_translation_model.keras")

In [18]:
with open('input_tokenizer.pkl', 'wb') as f:
    pickle.dump(input_tokenizer, f)
with open('target_tokenizer.pkl', 'wb') as f:
    pickle.dump(target_tokenizer, f)